# Predicting Student Course Completion — EdX Dataset
**Omar Ahmad**

A machine learning project analyzing student engagement data from an online education platform to predict whether a learner will earn course certification. The goal is to compare classification approaches, evaluate their tradeoffs, and surface insights relevant to educational outcomes.

---

## Project Overview

**Dataset:** EdX student activity and demographic data  
**Prediction target:** Whether a student earns course certification (`certified`)  
**Approach:** End-to-end pipeline — data loading, preprocessing, model comparison, hyperparameter tuning, and final prediction

**Models evaluated:**
- Logistic Regression (regularization sweep)
- Random Forest (depth and tree count sweep)

**Key techniques:** One-hot encoding, median imputation, train/validation split with stratification, confusion matrix analysis, accuracy comparison visualization

## 1. Load Data

In [ ]:
import requests

def save_file(url, file_name):
    r = requests.get(url)
    with open(file_name, 'wb') as f:
        f.write(r.content)

save_file('https://courses.cs.washington.edu/courses/cse416/23sp/homeworks/hw5/edx_train.csv', 'edx_train.csv')
save_file('https://courses.cs.washington.edu/courses/cse416/23sp/homeworks/hw5/edx_test.csv', 'edx_test.csv')
print("Data loaded successfully.")

## 2. Data Preprocessing & Feature Engineering

The dataset contains a mix of numerical and categorical features describing student demographics and engagement behavior. Key preprocessing steps:

- **Drop identifier column** (`userid_DI`) — not a meaningful predictor
- **Impute missing values** — median for numeric columns, placeholder for categoricals
- **One-hot encode** categorical variables so models can interpret them numerically
- **Stratified train/validation split** (80/20) to preserve class balance across both sets

Engagement-related features (time spent, chapters explored, interactions) are expected to be the strongest predictors of certification, since completion is more directly tied to participation than to demographics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

df_train = pd.read_csv("edx_train.csv")
df_test  = pd.read_csv("edx_test.csv")

target = "certified"
id_col = "userid_DI"

# Separate features and target
y = df_train[target].astype(int)
X = df_train.drop(columns=[target, id_col]).copy()

# Impute missing values
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

cat_cols = X.select_dtypes(include=["object"]).columns
X[cat_cols] = X[cat_cols].fillna("unknown")

# One-hot encode categorical columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=False)

# Stratified train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)} | Validation samples: {len(X_val)}")
print(f"Features: {X_train.shape[1]}")
print(f"Class balance (train): {y_train.value_counts().to_dict()}")

## 3. Model Comparison

Four configurations were evaluated — two regularization strengths for Logistic Regression and two complexity levels for Random Forest. Each model was trained on the training split and evaluated on the held-out validation set.

**Why these models?**
- *Logistic Regression* — linear baseline, interpretable, useful for understanding which features linearly predict certification
- *Random Forest* — ensemble of decision trees capable of capturing nonlinear patterns and feature interactions; well-suited for tabular data with mixed feature types

In [ ]:
experiments = [
    ("Logistic Regression  (C=1.0)", LogisticRegression(max_iter=2000, C=1.0)),
    ("Logistic Regression  (C=0.1)", LogisticRegression(max_iter=2000, C=0.1)),
    ("Random Forest  (depth=12, trees=300)", RandomForestClassifier(max_depth=12, n_estimators=300, random_state=42, n_jobs=-1)),
    ("Random Forest  (depth=None, trees=600)", RandomForestClassifier(max_depth=None, n_estimators=600, random_state=42, n_jobs=-1)),
]

results = []
best_model, best_name, best_val_acc = None, None, -1

for name, model in experiments:
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc   = accuracy_score(y_val,   model.predict(X_val))
    results.append({"Model": name, "Train Accuracy": round(train_acc, 4), "Val Accuracy": round(val_acc, 4)})
    if val_acc > best_val_acc:
        best_val_acc, best_model, best_name = val_acc, model, name

df_results = pd.DataFrame(results).sort_values("Val Accuracy", ascending=False)
print(df_results.to_string(index=False))
print(f"\nBest model: {best_name}  |  Validation accuracy: {best_val_acc:.4f}")

## 4. Visualizing Model Performance

Two visualizations help interpret model behavior:

1. **Train vs. Validation Accuracy** — reveals overfitting; a large gap between training and validation accuracy suggests the model memorized training patterns rather than generalizing
2. **Confusion Matrix (best model)** — breaks down true positives, false positives, true negatives, and false negatives to understand prediction errors beyond overall accuracy

In [ ]:
# Bar chart: Train vs Validation accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(df_results))
axes[0].bar(x - 0.2, df_results["Train Accuracy"], width=0.4, label="Train Accuracy", color="#2C5F8A")
axes[0].bar(x + 0.2, df_results["Val Accuracy"],   width=0.4, label="Validation Accuracy", color="#5BA3D9")
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_results["Model"], rotation=20, ha="right", fontsize=9)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Model Comparison: Train vs Validation Accuracy")
axes[0].legend()

# Confusion matrix: best model
best_val_pred = best_model.predict(X_val)
cm = confusion_matrix(y_val, best_val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Certified", "Certified"])
disp.plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title(f"Confusion Matrix\n({best_name})")

plt.tight_layout()
plt.show()

## 5. Findings & Model Interpretation

**Logistic Regression** performed consistently across regularization settings but plateaued at a lower accuracy, suggesting a linear decision boundary is insufficient for this dataset. Adjusting the regularization strength (`C`) had minimal impact on validation performance.

**Random Forest** significantly outperformed the linear baseline. Allowing deeper trees and more estimators improved validation accuracy, indicating the dataset contains nonlinear relationships and feature interactions that tree-based ensembles capture well. The best-performing configuration (unconstrained depth, 600 trees) showed some overfitting — its training accuracy was near-perfect — but still generalized well to the validation set.

**Feature importance insight:** Engagement-based features (course interactions, chapters explored, time spent) likely drive predictive performance more than demographic variables. This aligns with the intuition that certification is earned through participation, not determined by background characteristics.

**Selected model:** Random Forest (depth=None, trees=600) based on highest validation accuracy.

## 6. Retrain on Full Dataset & Generate Predictions

The best model is retrained on the complete labeled training data before generating predictions on the held-out test set. This maximizes the information available for learning while keeping preprocessing consistent between train and test.

In [ ]:
# Preprocess full training set
y_full = df_train[target].astype(int)
X_full = df_train.drop(columns=[target, id_col]).copy()

X_full[num_cols] = X_full[num_cols].fillna(X_full[num_cols].median())
X_full[cat_cols] = X_full[cat_cols].fillna("unknown")
X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

# Retrain best model on full training data
best_model.fit(X_full, y_full)

# Preprocess test set with same pipeline
X_test = df_test.drop(columns=[id_col]).copy()
X_test[num_cols] = X_test[num_cols].fillna(X_full[num_cols].median())
X_test[cat_cols] = X_test[cat_cols].fillna("unknown")
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=False)

# Align test columns to match training feature space
X_test = X_test.reindex(columns=X_full.columns, fill_value=0)

# Generate predictions
predictions = best_model.predict(X_test)

# Save output
output = df_test[[id_col]].copy()
output.loc[:, target] = predictions
output.to_csv("predictions.csv", index=False)

print(f"Predictions saved. Certified rate: {predictions.mean():.2%}")

## 7. Ethical Considerations

Predictive models built on student data carry real responsibilities. Several concerns are worth noting:

**Reflection of existing inequalities:** Students from certain countries or educational backgrounds may complete courses at higher rates due to better internet access, prior preparation, or financial stability — not inherent capability. A model trained on this data will learn those patterns. Decisions based on its output could reinforce existing disparities rather than address them.

**Correlation vs. causation:** The model identifies statistical associations between features and certification — it does not explain *why* students complete or drop out. Acting on correlations as if they were causes (e.g., deprioritizing students from underperforming demographic groups) could lead to discriminatory outcomes.

**Appropriate use:** The most constructive application of this model would be early identification of students at risk of not completing a course, enabling targeted support and intervention — not the inverse, where resources are concentrated only on students already likely to succeed.

Any production deployment of a model like this should include fairness audits across demographic groups and ongoing monitoring to detect disparate impact.